Version to run locally.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
from helpers.data_management import (DatasetConfig,
    create_LOSO_dataset_dataloader)
from helpers.constants import *
from helpers.visualization import (inspect_knee_moment_dataset, plot_loss,
                                   prediction_overlay, evaluate_visualize_model)
from helpers.modules import KneeCNN, KneeTCN, KneeLSTM
from helpers.running import train_model, evaluate_model, loso_cross_validation

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

batch_size = 512
num_epoches = 5
lr = 1e-3

# Dataset and Network Example

In [ ]:
leave_one_out_subject = SUBJECTS[2]

cfg = DatasetConfig(tasks = PERIODIC_TASK_PREFIXES)
train_dataset, test_dataset, train_dataloader, test_dataloader = \
    create_LOSO_dataset_dataloader(
        leave_one_out_subject, dataset_cfg=cfg, batch_size=batch_size)

print(len(train_dataset))
print(len(test_dataset))

iterator = iter(train_dataset) # only for visualization

In [ ]:
# visualize dataset
x,y = next(iterator)
_,_ = inspect_knee_moment_dataset(torch.squeeze(x),y)

In [ ]:
cnn_model = KneeCNN().to(device)
checkpoint = train_model(cnn_model, train_dataloader, test_dataloader,
                         num_epochs=10)
plot_loss(checkpoint['train_losses'], checkpoint['test_losses'])

In [ ]:
preds, targets, rmse, r2 = evaluate_model(cnn_model, test_dataloader,
                                          train_dataset.scaler_y)
print(f'RMSE: {rmse:.4f} Nm')
print(f'R\u00b2:   {r2:.4f}')
prediction_overlay(targets, preds, rmse, r2)

# LOSO Evaluation

### Change Architecture

In [ ]:
experiment_name = "base_"
# CNN
rmses, r2s, last_model = loso_cross_validation(
    KneeCNN,
    num_epoches = num_epoches,
    experiment_name = experiment_name,
    lr = lr,
    )
# predict and plot.
evaluate_visualize_model(
    last_model,
    SUBJECTS_SUBSET[-1],
    dataset_cfg=DatasetConfig(test_tasks = NON_PERIODIC_TASK_PREFIXES),
    )

In [ ]:
# TCN
rmses, r2s, last_model = loso_cross_validation(
    KneeTCN,
    num_epoches = num_epoches,
    experiment_name = experiment_name,
    lr = lr
    )
# predict and plot.
evaluate_visualize_model(
    last_model,
    SUBJECTS_SUBSET[-1],
    dataset_cfg=DatasetConfig(test_tasks = NON_PERIODIC_TASK_PREFIXES),
    )

In [ ]:
# LSTM
rmses, r2s, last_model = loso_cross_validation(
    KneeLSTM,
    dataset_cfg = DatasetConfig(full_horizon_output = True),
    num_epoches = num_epoches,
    experiment_name = experiment_name,
    lr = lr
    )
# predict and plot.
evaluate_visualize_model(
    last_model,
    SUBJECTS_SUBSET[-1],
    dataset_cfg=DatasetConfig(
        full_horizon_output = True,
        test_tasks = NON_PERIODIC_TASK_PREFIXES
        ),
    )

## Hyperparameter tuning

### Increasing hidden layer size (default is 64)

In [ ]:
experiment_name = "layer_"
# CNN, hidden size = 128.
rmses, r2s, last_model = loso_cross_validation(
    KneeCNN,
    num_epoches = num_epoches,
    hidden_layer_size = 128,
    experiment_name = experiment_name
    )
# predict and plot.
evaluate_visualize_model(
    last_model,
    SUBJECTS_SUBSET[-1],
    dataset_cfg=DatasetConfig(test_tasks = NON_PERIODIC_TASK_PREFIXES),
    )

In [ ]:
# TCN, hidden size = 128.
rmses, r2s, last_model = loso_cross_validation(
    KneeTCN,
    num_epoches = num_epoches,
    hidden_layer_size = 128,
    experiment_name = experiment_name
    )
# predict and plot.
evaluate_visualize_model(
    last_model,
    SUBJECTS_SUBSET[-1],
    dataset_cfg=DatasetConfig(test_tasks = NON_PERIODIC_TASK_PREFIXES),
    )

### Changng window size (default is 100)

In [ ]:
experiment_name = "window_"
# CNN
rmses, r2s, last_model = loso_cross_validation(
    KneeCNN,
    dataset_cfg = DatasetConfig(window_size = 100),
    num_epoches = num_epoches,
    experiment_name = experiment_name
    )
# predict and plot.
evaluate_visualize_model(
    last_model,
    SUBJECTS_SUBSET[-1],
    dataset_cfg=DatasetConfig(test_tasks = NON_PERIODIC_TASK_PREFIXES,
                              window_size = 100),
    )

In [ ]:
# TCN
rmses, r2s, last_model = loso_cross_validation(
    KneeTCN,
    dataset_cfg = DatasetConfig(window_size = 100),
    num_epoches = num_epoches,
    experiment_name = experiment_name
    )
# predict and plot.
evaluate_visualize_model(
    last_model,
    SUBJECTS_SUBSET[-1],
    dataset_cfg=DatasetConfig(test_tasks = NON_PERIODIC_TASK_PREFIXES,
                              window_size = 100),
    )

## Sensor Ablation Studies (TCN)

In [ ]:
# removing Shank IMU
rmses, r2s, last_model = loso_cross_validation(
    KneeTCN,
    dataset_cfg = DatasetConfig(ablated_sensors = ["imu_shank"]),
    num_epoches = num_epoches,
    experiment_name = "ablation_shank_"
    )
evaluate_visualize_model(
    last_model,
    SUBJECTS_SUBSET[-1],
    dataset_cfg=DatasetConfig(test_tasks = NON_PERIODIC_TASK_PREFIXES,
                              ablated_sensors = ["imu_shank"]),
    )

In [ ]:
# removing Shank IMU, knee
rmses, r2s, last_model = loso_cross_validation(
    KneeTCN,
    dataset_cfg = DatasetConfig(
        ablated_sensors = ["imu_shank", "angle", "velocity"]),
    num_epoches = num_epoches,
    experiment_name = "ablation_shank_knee_"
    )
evaluate_visualize_model(
    last_model,
    SUBJECTS_SUBSET[-1],
    dataset_cfg=DatasetConfig(test_tasks = NON_PERIODIC_TASK_PREFIXES,
                              ablated_sensors = ["imu_shank", "angle", "velocity"]),
    )

## Periodic vs non-Periodic

In [ ]:
rmses, r2s, last_model = loso_cross_validation(
    KneeTCN,
    dataset_cfg = DatasetConfig(tasks = PERIODIC_TASK_PREFIXES,
                                test_tasks = NON_PERIODIC_TASK_PREFIXES
                                ),
    num_epoches = num_epoches,
    experiment_name = "periodic_train_"
    )
evaluate_visualize_model(
    last_model,
    SUBJECTS_SUBSET[-1],
    dataset_cfg=DatasetConfig(tasks = PERIODIC_TASK_PREFIXES, # for scaler fitting
                              test_tasks = NON_PERIODIC_TASK_PREFIXES),
    )

In [ ]:
rmses, r2s, last_model = loso_cross_validation(
    KneeTCN,
    dataset_cfg = DatasetConfig(tasks = NON_PERIODIC_TASK_PREFIXES,
                                test_tasks = PERIODIC_TASK_PREFIXES
                                ),
    num_epoches = num_epoches,
    experiment_name = "nonperiodic_train_"
    )
evaluate_visualize_model(
    last_model,
    SUBJECTS_SUBSET[-1],
    dataset_cfg=DatasetConfig(tasks = NON_PERIODIC_TASK_PREFIXES, # for scaler fitting
                              test_tasks = PERIODIC_TASK_PREFIXES),
    )

# Zip and save models to drive

In [ ]:
!zip -r ./generated_data.zip ./generated_data
!cp -r ./generated_data.zip ../drive/MyDrive/Wearable_notebooks/